# Agentic AI - Turning functions into tools

### Import libraries and load env variables

In [1]:
import json
import utils
import aisuite as ai
from dotenv import load_dotenv
import functions as f

load_dotenv()

True

### Create AISuite Client

In [2]:
client = ai.Client()

### Define prompt that uses above function as tool and call LLM with get_current_tme as tool

In [3]:
prompt = "What time is it?"
messages = [{"role": "user", "content": prompt}]

response = client.chat.completions.create(
    model="openai:gpt-4o",
    messages=messages,
    tools=[f.get_current_time],
    tool_choice="auto",
    max_tokens=500,
    temperature=0,
    max_turns=5)

print(response.choices[0].message.content)

The current time is 09:00:16.


### Understand the calls happeening in background

In [4]:
utils.pretty_print_chat_completion(response)

### Manually define the tools

In [5]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_current_time",
        "description": "Returns the current time as a string.",
        "parameters": {}
    }
}]

In [6]:
response = client.chat.completions.create(
    model="openai:gpt-4o",
    messages=messages,
    tools=tools, # <-- Your list of tools with get_current_time
    # max_turns=5 # <-- When defining tools manually, you must handle calls yourself and cannot use max_turns
)

In [7]:
response2 = None

if response.choices[0].message.tool_calls:
    tool_call = response.choices[0].message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    print(tool_call)
    tool_response = f.get_current_time()
    messages.append(response.choices[0].message.to_dict())
    messages.append({
        "role": "tool", "tool_call_id": tool_call.id, "content": str(tool_response)
    })

    response2 = client.chat.completions.create(
        model="openai:gpt-4o",
        messages=messages,
        tools=tools
    )

    print(response2.choices[0].message.content)

ChatCompletionMessageFunctionToolCall(id='call_dlSnn0Uo8qUhd5VBtbIuhrL7', function=Function(arguments='{}', name='get_current_time'), type='function')
The current time is 09:00:17.


### Add more tools to LLM

In [8]:
prompt = "Can you get the weather for my location?"

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt
    )}],
    tools=[
        f.get_current_time,
        f.get_weather_from_ip,
        f.write_text_to_file,
        f.generate_qr_code
    ],
    max_turns=5
)

utils.pretty_print_chat_completion(response)

In [9]:
prompt = "Can you make a txt note for me called reminders.txt that reminds me to call Daniel tomorrow at 7PM?"

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt
    )}],
    tools=[
        f.get_current_time,
        f.get_weather_from_ip,
        f.write_text_to_file,
        f.generate_qr_code
    ],
    max_turns=5
)

utils.pretty_print_chat_completion(response)

In [10]:
prompt = "Can you make a QR code for me using my company's logo that goes to www.deeplearning.ai? " \
"The file 'logo.jpg' is located in the same directory as this notebook. " \
"Use exactly 'logo.jpg' for the image_path argument. " \
"Also write me a txt note with the current weather please." \
"Also Can you make a txt note for me called reminders.txt that reminds me to call Daniel tomorrow at 7PM?"

response = client.chat.completions.create(
    model="openai:o4-mini",
    messages=[{"role": "user", "content": (
        prompt
    )}],
    tools=[
        f.get_current_time,
        f.get_weather_from_ip,
        f.write_text_to_file,
        f.generate_qr_code
    ],
    max_turns=5
)

utils.pretty_print_chat_completion(response)

### Use ToolAgent class from utils.py

In [13]:
import toolagent

tools = [{
    "type": "function",
    "function": {
        "name": "get_current_time",
        "description": "Returns the current time as a string.",
        "parameters": {"type": "object", "properties": {}}
    }
}, {
    "type": "function",
    "function": {
        "name": "get_weather_from_ip",
        "description": "Returns the current weather for the user's location based on their IP address.",
        "parameters": {"type": "object", "properties": {}}
    }
}, {
    "type": "function",
    "function": {
        "name": "write_text_to_file",
        "description": "Writes the given text to a file with the specified filename.",
        "parameters": {
            "type": "object",
            "properties": {
                "content": {"type": "string", "description": "The text content to write into the file."},                
                "file_path": {"type": "string", "description": "The path to write file to."}
            },
            "required": ["content", "file_path"]
        }
    }
}, {
    "type": "function",
    "function": {
        "name": "generate_qr_code",
        "description": "Generates a QR code image with the specified data and saves it to the given image path.",
        "parameters": {
            "type": "object",
            "properties": {
                "data": {"type": "string", "description": "The data to encode in the QR code."},
                "filename": {"type": "string", "description": "The name of the file to write to."},
                "image_path": {"type": "string", "description": "The file path where the QR code image will be saved."}
            },
            "required": ["data", "filename", "image_path"]
        }
    }
}]

agent = toolagent.ToolAgent(
    client=client,
    model="openai:gpt-4o",
    tools=tools,
    available_funcs={
        "get_current_time": f.get_current_time,
        "get_weather_from_ip": f.get_weather_from_ip,
        "write_text_to_file": f.write_text_to_file,
        "generate_qr_code": f.generate_qr_code
    }
)

agent.enable_inspector(True).enable_protocol_dump(True)

response = agent.ask(prompt)

print("Assistant:", response)


### PROTOCOL [REQUEST] ###
{
  "model": "openai:gpt-4o",
  "messages": [
    {
      "role": "user",
      "content": "Can you make a QR code for me using my company's logo that goes to www.deeplearning.ai? The file 'logo.jpg' is located in the same directory as this notebook. Use exactly 'logo.jpg' for the image_path argument. Also write me a txt note with the current weather please.Also Can you make a txt note for me called reminders.txt that reminds me to call Daniel tomorrow at 7PM?"
    }
  ],
  "tools": [
    {
      "type": "function",
      "function": {
        "name": "get_current_time",
        "description": "Returns the current time as a string.",
        "parameters": {
          "type": "object",
          "properties": {}
        }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "get_weather_from_ip",
        "description": "Returns the current weather for the user's location based on their IP address.",
        "parameters": {
      